# 11b - Binary Two-Phase Cancer-Risk Model: Final Evaluation

**Pipeline:** Skin Cancer Binary Cancer-Risk Screening
**Purpose:** Load `best_model_binary_2phase.pt`, select decision thresholds on the
validation set, then evaluate once on the held-out test set.

**Binary mapping:**
- NV  = 0 = non_cancer
- MEL = 1 = cancer_risk
- BCC = 1 = cancer_risk

**Rules:**
- No training, no weight updates.
- Validation set is used **only** for threshold selection.
- Test set is used **only once** for final reporting.
- No image/manifest/split modifications.
- This notebook targets `best_model_binary_2phase.pt` only.

> **Medical note:** This model estimates cancer-risk probability for screening support.
> It is not a clinical diagnosis.

---


## Section 0 - Imports and Environment

In [1]:
import json
import os
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    recall_score,
    roc_auc_score,
    roc_curve,
)

CUDA_AVAILABLE = torch.cuda.is_available()
try:
    DEVICE   = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "N/A"
except AssertionError:
    DEVICE, GPU_NAME, CUDA_AVAILABLE = torch.device("cpu"), "N/A", False

print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"CUDA        : {CUDA_AVAILABLE}")
print(f"GPU         : {GPU_NAME}")
print(f"Device      : {DEVICE}")


torch       : 2.6.0+cu124
torchvision : 0.21.0+cu124
CUDA        : True
GPU         : NVIDIA GeForce RTX 4060 Laptop GPU
Device      : cuda


## Section 1 - Paths and Configuration

In [2]:
OUTPUT_ROOT   = Path(r"C:\SKIN CANCER v2\pipe output")
PREPROC_DIR   = OUTPUT_ROOT / "preprocessing"
TRAIN_2PH_DIR = OUTPUT_ROOT / "pytorch_training_binary_2phase"
EVAL_DIR      = OUTPUT_ROOT / "pytorch_binary_2phase_evaluation"

MODEL_PATH    = TRAIN_2PH_DIR / "best_model_binary_2phase.pt"

BINARY_NAMES = ["non_cancer", "cancer_risk"]
BINARY_INDEX = {"NV": 0, "MEL": 1, "BCC": 1}
CLASS_NAMES  = ["NV", "MEL", "BCC"]

EXPECTED_TEST = {
    "total":       3045,
    "NV":          1894,
    "MEL":          639,
    "BCC":          512,
    "non_cancer":  1894,
    "cancer_risk": 1151,
}

IMG_SIZE    = 224
RESIZE_TO   = 256
BATCH_SIZE  = 32
NUM_WORKERS = 0
PIN_MEMORY  = CUDA_AVAILABLE

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Threshold grid for selection
THRESH_GRID = np.round(np.arange(0.01, 1.00, 0.01), 3)
HIGH_RECALL_TARGET = 0.90

print("Config loaded.")
print(f"  Model path : {MODEL_PATH}")
print(f"  Model exists: {MODEL_PATH.exists()}")


Config loaded.
  Model path : C:\SKIN CANCER v2\pipe output\pytorch_training_binary_2phase\best_model_binary_2phase.pt
  Model exists: True


## Section 2 - Create Output Folder

In [3]:
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {EVAL_DIR}")


Ready: C:\SKIN CANCER v2\pipe output\pytorch_binary_2phase_evaluation


## Section 3 - Rebuild Model Architecture and Load Checkpoint

In [4]:
def build_model():
    """Exact same architecture as 10_pytorch_binary_two_phase_training.ipynb."""
    model = efficientnet_b0(weights=None)
    in_feat = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_feat, 2),
    )
    return model.to(DEVICE)


if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {MODEL_PATH}\n"
        "Run 10_pytorch_binary_two_phase_training.ipynb first."
    )

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model = build_model()
model.load_state_dict(checkpoint["model_state"])
model.eval()

ckpt_epoch   = checkpoint.get("global_epoch", checkpoint.get("epoch", "?"))
ckpt_phase   = checkpoint.get("phase", "?")
ckpt_pr_auc  = checkpoint.get("best_pr_auc", "?")
ckpt_metrics = checkpoint.get("val_metrics", {})

print(f"Checkpoint loaded:")
print(f"  file          : {MODEL_PATH.name}")
print(f"  global_epoch  : {ckpt_epoch}")
print(f"  phase         : {ckpt_phase}")
print(f"  best_val_pr_auc : {ckpt_pr_auc}")
if ckpt_metrics:
    for k, v in ckpt_metrics.items():
        print(f"  {k:<28}: {v}")
print(f"\nModel parameter count: {sum(p.numel() for p in model.parameters()):,}")


Checkpoint loaded:
  file          : best_model_binary_2phase.pt
  global_epoch  : 17
  phase         : phase2_finetune
  best_val_pr_auc : 0.90232
  val_loss                    : 0.35803
  val_accuracy                : 0.85591
  val_precision_cancer        : 0.80245
  val_recall_cancer           : 0.81528
  val_f1_cancer               : 0.80881
  val_roc_auc                 : 0.92955
  val_pr_auc                  : 0.90232

Model parameter count: 4,010,110


## Section 4 - Load Manifests

In [5]:
dfs = {}
for split in ["val", "test"]:
    p = PREPROC_DIR / f"{split}_manifest_preprocessed.csv"
    if not p.exists():
        raise FileNotFoundError(f"Manifest not found: {p}")
    dfs[split] = pd.read_csv(p, low_memory=False)
    dfs[split]["binary_label"] = dfs[split]["final_authoritative_label"].map(BINARY_INDEX)
    print(f"Loaded {split}: {len(dfs[split]):,} rows")

# Count check (test)
test_df = dfs["test"]
cc = test_df["final_authoritative_label"].value_counts()
bc = test_df["binary_label"].value_counts()
print("\n=== TEST COUNT CHECK ===")
print(f"  total      : {len(test_df):,}  "
      f"{'OK' if len(test_df)==EXPECTED_TEST['total'] else 'DIFF'}")
for cls in CLASS_NAMES:
    n = int(cc.get(cls, 0))
    e = EXPECTED_TEST[cls]
    print(f"  {cls:<6}     : {n:,}  {'OK' if n==e else f'DIFF exp={e}'}")
print(f"  non_cancer : {int(bc.get(0,0)):,}  "
      f"{'OK' if int(bc.get(0,0))==EXPECTED_TEST['non_cancer'] else 'DIFF'}")
print(f"  cancer_risk: {int(bc.get(1,0)):,}  "
      f"{'OK' if int(bc.get(1,0))==EXPECTED_TEST['cancer_risk'] else 'DIFF'}")

# Load train manifest for reference only (not used in evaluation)
train_ref = pd.read_csv(PREPROC_DIR / "train_manifest_preprocessed.csv", low_memory=False)
print(f"\nTrain manifest loaded for reference: {len(train_ref):,} rows (not used in evaluation)")
del train_ref


Loaded val: 3,012 rows
Loaded test: 3,045 rows

=== TEST COUNT CHECK ===
  total      : 3,045  OK
  NV         : 1,894  OK
  MEL        : 639  OK
  BCC        : 512  OK
  non_cancer : 1,894  OK
  cancer_risk: 1,151  OK

Train manifest loaded for reference: 14,332 rows (not used in evaluation)


## Section 5 - Transforms and Dataset

Inference-only transform: Resize(256) → CenterCrop(224) → ToTensor → Normalize.
No augmentation.


In [6]:
eval_transform = T.Compose([
    T.Resize(RESIZE_TO),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class SkinLesionDataset(Dataset):
    BINARY = {"NV": 0, "MEL": 1, "BCC": 1}

    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(str(row["preprocessed_full_path"])).convert("RGB")
        if self.transform:
            img = self.transform(img)
        lbl = self.BINARY.get(str(row["final_authoritative_label"]), -1)
        return img, torch.tensor(lbl, dtype=torch.long)


# Smoke test
_ds = SkinLesionDataset(dfs["val"].head(2), eval_transform)
_img, _lbl = _ds[0]
print(f"Dataset smoke test: img={_img.shape}  label={_lbl.item()}")
del _ds, _img, _lbl


Dataset smoke test: img=torch.Size([3, 224, 224])  label=0


## Section 6 - Run Inference on Validation and Test Sets

In [7]:
@torch.no_grad()
def run_inference(model, df, transform):
    """Returns (cancer_risk_probs, true_binary_labels) aligned with df rows."""
    ds  = SkinLesionDataset(df, transform)
    ldr = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)
    all_probs, all_labels = [], []
    model.eval()
    for imgs, labels in ldr:
        imgs = imgs.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=CUDA_AVAILABLE):
            out = model(imgs)
        probs = torch.softmax(out, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_probs, dtype=np.float32), np.array(all_labels, dtype=np.int32)


print("Running inference on validation set...")
val_probs, val_true = run_inference(model, dfs["val"], eval_transform)
print(f"  done: {len(val_probs):,} samples")

print("Running inference on test set...")
test_probs, test_true = run_inference(model, dfs["test"], eval_transform)
print(f"  done: {len(test_probs):,} samples")

# Sanity check alignment
assert len(val_probs)  == len(dfs["val"]),  "Val length mismatch"
assert len(test_probs) == len(dfs["test"]), "Test length mismatch"

print(f"\nVal  — prob range [{val_probs.min():.4f}, {val_probs.max():.4f}]  "
      f"mean={val_probs.mean():.4f}")
print(f"Test — prob range [{test_probs.min():.4f}, {test_probs.max():.4f}]  "
      f"mean={test_probs.mean():.4f}")

# Global AUC metrics (threshold-independent)
val_roc_auc = float(roc_auc_score(val_true, val_probs))
val_pr_auc  = float(average_precision_score(val_true, val_probs, pos_label=1))
test_roc_auc = float(roc_auc_score(test_true, test_probs))
test_pr_auc  = float(average_precision_score(test_true, test_probs, pos_label=1))

print(f"\nVal  ROC-AUC={val_roc_auc:.5f}  PR-AUC={val_pr_auc:.5f}")
print(f"Test ROC-AUC={test_roc_auc:.5f}  PR-AUC={test_pr_auc:.5f}")


Running inference on validation set...
  done: 3,012 samples
Running inference on test set...
  done: 3,045 samples

Val  — prob range [0.0000, 1.0000]  mean=0.3983
Test — prob range [0.0000, 1.0000]  mean=0.4183

Val  ROC-AUC=0.92952  PR-AUC=0.90180
Test ROC-AUC=0.92758  PR-AUC=0.90013


## Section 7 - Save Prediction CSVs

In [8]:
def build_pred_df(df, probs, true_labels, split_name):
    out = df[[c for c in [
        "preprocessed_full_path", "original_full_path",
        "final_authoritative_label", "binary_label",
        "canonical_match_id", "lesion_id",
    ] if c in df.columns]].copy().reset_index(drop=True)
    out["split"]                = split_name
    out["binary_true_label"]    = true_labels
    out["cancer_risk_probability"] = np.round(probs, 6)
    out["predicted_binary_default"]    = (probs >= 0.50).astype(int)
    return out

val_pred_df  = build_pred_df(dfs["val"],  val_probs,  val_true,  "val")
test_pred_df = build_pred_df(dfs["test"], test_probs, test_true, "test")

val_pred_path  = EVAL_DIR / "binary_2phase_val_predictions.csv"
test_pred_path = EVAL_DIR / "binary_2phase_test_predictions.csv"
val_pred_df.to_csv(val_pred_path,   index=False)
test_pred_df.to_csv(test_pred_path, index=False)
print(f"Saved {val_pred_path.name}   ({len(val_pred_df):,} rows)")
print(f"Saved {test_pred_path.name}  ({len(test_pred_df):,} rows)")


Saved binary_2phase_val_predictions.csv   (3,012 rows)
Saved binary_2phase_test_predictions.csv  (3,045 rows)


## Section 8 - Threshold Selection on Validation Set

Four thresholds derived from validation probabilities only:
1. **default** — 0.50
2. **max_f1** — maximises cancer F1 on val
3. **youden_j** — maximises Youden J (sensitivity + specificity − 1) on val
4. **high_recall** — highest threshold that still achieves cancer recall ≥ 0.90 on val


In [9]:
def val_metrics_at_thresh(probs, true_labels, thresh):
    preds = (probs >= thresh).astype(int)
    cm = confusion_matrix(true_labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    acc  = accuracy_score(true_labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        true_labels, preds, labels=[0, 1], average=None, zero_division=0)
    cancer_prec = float(prec[1]) if len(prec) > 1 else 0.0
    cancer_rec  = float(rec[1])  if len(rec)  > 1 else 0.0
    cancer_f1   = float(f1[1])   if len(f1)   > 1 else 0.0
    spec = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
    return {
        "threshold":       round(float(thresh), 4),
        "accuracy":        round(acc, 5),
        "cancer_precision": round(cancer_prec, 5),
        "cancer_recall":   round(cancer_rec, 5),
        "specificity":     round(spec, 5),
        "cancer_f1":       round(cancer_f1, 5),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
    }


# ── Youden J via ROC curve ────────────────────────────────────────────────────
fpr_arr, tpr_arr, roc_thresholds = roc_curve(val_true, val_probs)
j_scores     = tpr_arr - fpr_arr
best_j_idx   = int(np.argmax(j_scores))
youden_thresh = float(np.clip(roc_thresholds[best_j_idx], 0.01, 0.99))

# ── Max F1 via grid search ────────────────────────────────────────────────────
f1_vals = []
for t in THRESH_GRID:
    m = val_metrics_at_thresh(val_probs, val_true, t)
    f1_vals.append(m["cancer_f1"])
f1_arr      = np.array(f1_vals)
max_f1_thresh = float(THRESH_GRID[np.argmax(f1_arr)])

# ── High-recall: highest threshold achieving >= 0.90 cancer recall ────────────
high_recall_thresh = 0.50
for t in sorted(THRESH_GRID, reverse=True):
    preds = (val_probs >= t).astype(int)
    rec = recall_score(val_true, preds, pos_label=1, zero_division=0)
    if rec >= HIGH_RECALL_TARGET:
        high_recall_thresh = float(t)
        break

THRESHOLDS = {
    "default_050":   0.50,
    "max_f1":        round(max_f1_thresh, 4),
    "youden_j":      round(youden_thresh, 4),
    "high_recall":   round(high_recall_thresh, 4),
}

print("=== SELECTED THRESHOLDS (from validation set) ===")
for name, t in THRESHOLDS.items():
    m = val_metrics_at_thresh(val_probs, val_true, t)
    print(f"  {name:<16} = {t:.4f}  "
          f"recall={m['cancer_recall']:.4f}  "
          f"prec={m['cancer_precision']:.4f}  "
          f"f1={m['cancer_f1']:.4f}  "
          f"spec={m['specificity']:.4f}  "
          f"FN={m['FN']}")

# Main threshold for plots and primary reporting
MAIN_THRESH_NAME = "youden_j"
MAIN_THRESH      = THRESHOLDS[MAIN_THRESH_NAME]
print(f"\nMain threshold for confusion matrix: {MAIN_THRESH_NAME} = {MAIN_THRESH:.4f}")


=== SELECTED THRESHOLDS (from validation set) ===
  default_050      = 0.5000  recall=0.8153  prec=0.8032  f1=0.8092  spec=0.8807  FN=208
  max_f1           = 0.6300  recall=0.7824  prec=0.8415  f1=0.8109  spec=0.9120  FN=245
  youden_j         = 0.4219  recall=0.8401  prec=0.7825  f1=0.8103  spec=0.8606  FN=180
  high_recall      = 0.1700  recall=0.9023  prec=0.6720  f1=0.7703  spec=0.7370  FN=110

Main threshold for confusion matrix: youden_j = 0.4219


In [10]:
# Save threshold selection table (val metrics at all thresholds in grid)
thresh_rows = []
for t in THRESH_GRID:
    m = val_metrics_at_thresh(val_probs, val_true, t)
    thresh_rows.append(m)
thresh_df = pd.DataFrame(thresh_rows)
thresh_df.insert(0, "val_roc_auc", round(val_roc_auc, 5))
thresh_df.insert(0, "val_pr_auc",  round(val_pr_auc,  5))

# Mark selected thresholds
thresh_df["selected"] = thresh_df["threshold"].apply(
    lambda t: next((k for k, v in THRESHOLDS.items() if abs(v - t) < 1e-5), ""))

thresh_path = EVAL_DIR / "binary_2phase_threshold_selection.csv"
thresh_df.to_csv(thresh_path, index=False)
print(f"Saved {thresh_path.name}  ({len(thresh_df)} threshold rows)")
print("\nSelected rows:")
sel = thresh_df[thresh_df["selected"] != ""]
print(sel[["threshold","selected","cancer_recall","specificity",
           "cancer_precision","cancer_f1","FN"]].to_string(index=False))


Saved binary_2phase_threshold_selection.csv  (99 threshold rows)

Selected rows:
 threshold    selected  cancer_recall  specificity  cancer_precision  cancer_f1  FN
      0.17 high_recall        0.90231      0.73701           0.67196    0.77028 110
      0.50 default_050        0.81528      0.88070           0.80315    0.80917 208
      0.63      max_f1        0.78242      0.91198           0.84145    0.81086 245


## Section 9 - Final Test Evaluation at All Thresholds

Test set evaluated at all four thresholds selected from validation.


In [11]:
def test_metrics_at_thresh(probs, true_labels, thresh, thresh_name):
    preds = (probs >= thresh).astype(int)
    cm = confusion_matrix(true_labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    acc  = accuracy_score(true_labels, preds)
    bacc = balanced_accuracy_score(true_labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        true_labels, preds, labels=[0, 1], average=None, zero_division=0)
    cancer_prec = float(prec[1]) if len(prec) > 1 else 0.0
    cancer_rec  = float(rec[1])  if len(rec)  > 1 else 0.0
    cancer_f1   = float(f1[1])   if len(f1)   > 1 else 0.0
    spec = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
    return {
        "threshold_name":    thresh_name,
        "threshold":         round(float(thresh), 4),
        "accuracy":          round(acc,  5),
        "balanced_accuracy": round(bacc, 5),
        "cancer_precision":  round(cancer_prec, 5),
        "cancer_recall":     round(cancer_rec,  5),
        "specificity":       round(spec,        5),
        "cancer_f1":         round(cancer_f1,   5),
        "roc_auc":           round(test_roc_auc, 5),
        "pr_auc":            round(test_pr_auc,  5),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "missed_cancer_count": int(fn),
        "false_positive_nv":   int(fp),
    }


test_metrics_rows = []
for thresh_name, thresh in THRESHOLDS.items():
    row = test_metrics_at_thresh(test_probs, test_true, thresh, thresh_name)
    test_metrics_rows.append(row)

test_metrics_df = pd.DataFrame(test_metrics_rows)

print("=== TEST SET METRICS AT ALL THRESHOLDS ===")
display_cols = ["threshold_name","threshold","cancer_recall","specificity",
                "cancer_precision","cancer_f1","balanced_accuracy",
                "roc_auc","pr_auc","FN","FP","TN","TP"]
from IPython.display import display
display(test_metrics_df[display_cols])

test_metrics_path = EVAL_DIR / "binary_2phase_test_metrics.csv"
test_metrics_df.to_csv(test_metrics_path, index=False)
print(f"\nSaved {test_metrics_path.name}")


=== TEST SET METRICS AT ALL THRESHOLDS ===


,threshold_name,threshold,cancer_recall,specificity,cancer_precision,cancer_f1,balanced_accuracy,roc_auc,pr_auc,FN,FP,TN,TP
0,default_050,0.5000,0.82711,0.86959,0.79399,0.81021,0.84835,0.92758,0.90013,199,247,1647,952
1,max_f1,0.6300,0.80191,0.89810,0.82706,0.81429,0.85001,0.92758,0.90013,228,193,1701,923
2,youden_j,0.4219,0.84796,0.84900,0.77338,0.80895,0.84848,0.92758,0.90013,175,286,1608,976
3,high_recall,0.1700,0.92007,0.71700,0.66395,0.77130,0.81854,0.92758,0.90013,92,536,1358,1059



Saved binary_2phase_test_metrics.csv


## Section 10 - Medical Error Analysis (Missed Cancer Cases)

At the main threshold (Youden J), identify:
1. All test samples where true label = cancer_risk but predicted = non_cancer (FN).
2. Subset: missed MELanomas.
3. Subset: missed BCC cases.


In [12]:
main_preds = (test_probs >= MAIN_THRESH).astype(int)

test_err_df = dfs["test"][[c for c in [
    "preprocessed_full_path", "original_full_path",
    "final_authoritative_label", "binary_label",
    "canonical_match_id", "lesion_id",
] if c in dfs["test"].columns]].copy().reset_index(drop=True)
test_err_df["binary_true_label"]      = test_true
test_err_df["cancer_risk_probability"] = np.round(test_probs, 6)
test_err_df["predicted_binary_label"]  = main_preds
test_err_df["threshold_used"]         = MAIN_THRESH
test_err_df["threshold_name"]         = MAIN_THRESH_NAME
test_err_df["split_assignment"]       = "test"

# Missed cancer cases: true=1, predicted=0
missed_mask = (test_err_df["binary_true_label"] == 1) & (test_err_df["predicted_binary_label"] == 0)
missed_df   = test_err_df[missed_mask].copy()
missed_mel  = missed_df[missed_df["final_authoritative_label"] == "MEL"].copy()
missed_bcc  = missed_df[missed_df["final_authoritative_label"] == "BCC"].copy()

print(f"At threshold={MAIN_THRESH:.4f} ({MAIN_THRESH_NAME}):")
print(f"  Total test cancer_risk cases : {int(test_true.sum()):,}")
print(f"  Missed cancer cases (FN)     : {len(missed_df):,}")
print(f"  Missed MELanomas             : {len(missed_mel):,}")
print(f"  Missed BCC cases             : {len(missed_bcc):,}")
print(f"  Cancer recall                : {1 - len(missed_df)/max(int(test_true.sum()),1):.4f}")

# Save error files
missed_path     = EVAL_DIR / "binary_2phase_missed_cancer_cases.csv"
missed_mel_path = EVAL_DIR / "binary_2phase_missed_melanomas.csv"
missed_bcc_path = EVAL_DIR / "binary_2phase_missed_bcc_cases.csv"
missed_df.to_csv(missed_path,     index=False)
missed_mel.to_csv(missed_mel_path, index=False)
missed_bcc.to_csv(missed_bcc_path, index=False)
print(f"\nSaved {missed_path.name}      ({len(missed_df)} rows)")
print(f"Saved {missed_mel_path.name}  ({len(missed_mel)} rows)")
print(f"Saved {missed_bcc_path.name}  ({len(missed_bcc)} rows)")

# Add missed counts to test_metrics_df
for i, row in test_metrics_df.iterrows():
    t = row["threshold"]
    preds_t = (test_probs >= t).astype(int)
    missed_t = dfs["test"]["final_authoritative_label"].reset_index(drop=True)
    fn_mask = (test_true == 1) & (preds_t == 0)
    test_metrics_df.at[i, "missed_mel_count"] = int((missed_t[fn_mask] == "MEL").sum())
    test_metrics_df.at[i, "missed_bcc_count"] = int((missed_t[fn_mask] == "BCC").sum())

test_metrics_df.to_csv(test_metrics_path, index=False)  # overwrite with updated counts
print(f"\nUpdated {test_metrics_path.name} with missed_mel_count and missed_bcc_count")


At threshold=0.4219 (youden_j):
  Total test cancer_risk cases : 1,151
  Missed cancer cases (FN)     : 175
  Missed MELanomas             : 148
  Missed BCC cases             : 27
  Cancer recall                : 0.8480

Saved binary_2phase_missed_cancer_cases.csv      (175 rows)
Saved binary_2phase_missed_melanomas.csv  (148 rows)
Saved binary_2phase_missed_bcc_cases.csv  (27 rows)

Updated binary_2phase_test_metrics.csv with missed_mel_count and missed_bcc_count


## Section 11 - Plots (Confusion Matrix, ROC, PR Curve)

In [13]:
# ── Confusion matrix at main threshold ───────────────────────────────────────
cm = confusion_matrix(test_true, main_preds, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["True: non_cancer", "True: cancer_risk"],
    columns=["Pred: non_cancer", "Pred: cancer_risk"],
)
cm_csv_path = EVAL_DIR / "binary_2phase_confusion_matrix.csv"
cm_df.to_csv(cm_csv_path)
print(f"Saved {cm_csv_path.name}")

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", linewidths=0.5,
    xticklabels=["non_cancer", "cancer_risk"],
    yticklabels=["non_cancer", "cancer_risk"],
    ax=ax,
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title(
    f"Confusion Matrix — Test Set\n"
    f"Threshold: {MAIN_THRESH:.4f} ({MAIN_THRESH_NAME})  "
    f"ROC-AUC={test_roc_auc:.4f}  PR-AUC={test_pr_auc:.4f}",
    fontsize=10,
)
plt.tight_layout()
cm_png = EVAL_DIR / "binary_2phase_confusion_matrix.png"
plt.savefig(cm_png, dpi=120)
plt.close()
print(f"Saved {cm_png.name}")


Saved binary_2phase_confusion_matrix.csv
Saved binary_2phase_confusion_matrix.png


In [14]:
# ── ROC curve ─────────────────────────────────────────────────────────────────
fpr_test, tpr_test, roc_t = roc_curve(test_true, test_probs)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_test, tpr_test, lw=2, color="tab:blue",
        label=f"Test ROC (AUC = {test_roc_auc:.4f})")
ax.plot([0, 1], [0, 1], lw=1, ls="--", color="gray", label="Random")

# Mark selected thresholds on ROC
thresh_colors = {"default_050": "black", "max_f1": "tab:orange",
                 "youden_j": "tab:red", "high_recall": "tab:green"}
for name, t in THRESHOLDS.items():
    preds_t = (test_probs >= t).astype(int)
    cm_t = confusion_matrix(test_true, preds_t, labels=[0, 1])
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    fpr_pt = fp_t / max(fp_t + tn_t, 1)
    tpr_pt = tp_t / max(tp_t + fn_t, 1)
    ax.scatter(fpr_pt, tpr_pt, s=80, zorder=5,
               color=thresh_colors.get(name, "purple"),
               label=f"{name} (t={t:.2f})", edgecolors="k", linewidths=0.5)

ax.set_xlabel("False Positive Rate (1 − Specificity)", fontsize=12)
ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=12)
ax.set_title("ROC Curve — Test Set (Binary Cancer-Risk)", fontsize=12)
ax.legend(fontsize=8, loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
roc_png = EVAL_DIR / "binary_2phase_roc_curve.png"
plt.savefig(roc_png, dpi=120)
plt.close()
print(f"Saved {roc_png.name}")


Saved binary_2phase_roc_curve.png


In [15]:
# ── PR curve ──────────────────────────────────────────────────────────────────
prec_arr, rec_arr, pr_t = precision_recall_curve(test_true, test_probs, pos_label=1)
baseline = float(test_true.sum()) / len(test_true)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(rec_arr, prec_arr, lw=2, color="tab:green",
        label=f"Test PR (AP = {test_pr_auc:.4f})")
ax.axhline(baseline, lw=1, ls="--", color="gray",
           label=f"Baseline (prevalence = {baseline:.3f})")

# Mark selected thresholds on PR
for name, t in THRESHOLDS.items():
    m = test_metrics_at_thresh(test_probs, test_true, t, name)
    ax.scatter(m["cancer_recall"], m["cancer_precision"], s=80, zorder=5,
               color=thresh_colors.get(name, "purple"),
               label=f"{name} (t={t:.2f})", edgecolors="k", linewidths=0.5)

ax.set_xlabel("Recall (Cancer Sensitivity)", fontsize=12)
ax.set_ylabel("Precision (Cancer PPV)", fontsize=12)
ax.set_title("Precision-Recall Curve — Test Set (cancer_risk class)", fontsize=12)
ax.legend(fontsize=8, loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
pr_png = EVAL_DIR / "binary_2phase_pr_curve.png"
plt.savefig(pr_png, dpi=120)
plt.close()
print(f"Saved {pr_png.name}")


Saved binary_2phase_pr_curve.png


## Section 12 - Evaluation Summary CSV

In [16]:
# Metrics at all thresholds already in test_metrics_df.
# Build a condensed summary record.
youden_row = test_metrics_df[test_metrics_df["threshold_name"] == "youden_j"].iloc[0]
hr_row     = test_metrics_df[test_metrics_df["threshold_name"] == "high_recall"].iloc[0]

summary_df = pd.DataFrame([{
    "model":                   "binary_2phase",
    "checkpoint":              MODEL_PATH.name,
    "ckpt_global_epoch":       ckpt_epoch,
    "ckpt_phase":              ckpt_phase,
    "ckpt_best_val_pr_auc":    ckpt_pr_auc,
    "val_roc_auc":             round(val_roc_auc, 5),
    "val_pr_auc":              round(val_pr_auc,  5),
    "test_roc_auc":            round(test_roc_auc, 5),
    "test_pr_auc":             round(test_pr_auc,  5),
    "thresh_default":          THRESHOLDS["default_050"],
    "thresh_max_f1":           THRESHOLDS["max_f1"],
    "thresh_youden_j":         THRESHOLDS["youden_j"],
    "thresh_high_recall":      THRESHOLDS["high_recall"],
    # Youden J results
    "youden_cancer_recall":    youden_row["cancer_recall"],
    "youden_specificity":      youden_row["specificity"],
    "youden_cancer_precision": youden_row["cancer_precision"],
    "youden_cancer_f1":        youden_row["cancer_f1"],
    "youden_balanced_acc":     youden_row["balanced_accuracy"],
    "youden_missed_cancer":    int(youden_row["missed_cancer_count"]),
    "youden_missed_mel":       int(youden_row.get("missed_mel_count", -1)),
    "youden_missed_bcc":       int(youden_row.get("missed_bcc_count", -1)),
    # High-recall results
    "hr_cancer_recall":        hr_row["cancer_recall"],
    "hr_specificity":          hr_row["specificity"],
    "hr_cancer_precision":     hr_row["cancer_precision"],
    "hr_cancer_f1":            hr_row["cancer_f1"],
    "hr_missed_cancer":        int(hr_row["missed_cancer_count"]),
    "hr_missed_mel":           int(hr_row.get("missed_mel_count", -1)),
    "hr_missed_bcc":           int(hr_row.get("missed_bcc_count", -1)),
    # Confirmation flags
    "training_performed":      False,
    "val_used_for":            "threshold_selection_only",
    "test_used_for":           "final_evaluation_only",
    "images_modified":         False,
    "manifests_modified":      False,
}])
summary_path = EVAL_DIR / "binary_2phase_evaluation_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved {summary_path.name}")
from IPython.display import display
display(summary_df.T)


Saved binary_2phase_evaluation_summary.csv


,0
model,binary_2phase
checkpoint,best_model_binary_2phase.pt
ckpt_global_epoch,17
ckpt_phase,phase2_finetune
ckpt_best_val_pr_auc,0.90232
val_roc_auc,0.92952
val_pr_auc,0.9018
test_roc_auc,0.92758
test_pr_auc,0.90013
thresh_default,0.5


## Section 13 - Output File Verification

In [17]:
required_files = [
    EVAL_DIR / "binary_2phase_val_predictions.csv",
    EVAL_DIR / "binary_2phase_test_predictions.csv",
    EVAL_DIR / "binary_2phase_threshold_selection.csv",
    EVAL_DIR / "binary_2phase_test_metrics.csv",
    EVAL_DIR / "binary_2phase_confusion_matrix.csv",
    EVAL_DIR / "binary_2phase_confusion_matrix.png",
    EVAL_DIR / "binary_2phase_roc_curve.png",
    EVAL_DIR / "binary_2phase_pr_curve.png",
    EVAL_DIR / "binary_2phase_missed_cancer_cases.csv",
    EVAL_DIR / "binary_2phase_missed_melanomas.csv",
    EVAL_DIR / "binary_2phase_missed_bcc_cases.csv",
    EVAL_DIR / "binary_2phase_evaluation_summary.csv",
]
print("Output file verification:")
all_ok = True
for p in required_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {p.name:<50} {size:>12,} bytes")
    if not exists:
        all_ok = False
print("\nAll required files present." if all_ok else "\nWARNING: missing files above.")


Output file verification:
  [OK] binary_2phase_val_predictions.csv                       437,571 bytes
  [OK] binary_2phase_test_predictions.csv                      445,892 bytes
  [OK] binary_2phase_threshold_selection.csv                     8,683 bytes
  [OK] binary_2phase_test_metrics.csv                              660 bytes
  [OK] binary_2phase_confusion_matrix.csv                           91 bytes
  [OK] binary_2phase_confusion_matrix.png                       34,795 bytes
  [OK] binary_2phase_roc_curve.png                              58,876 bytes
  [OK] binary_2phase_pr_curve.png                               53,189 bytes
  [OK] binary_2phase_missed_cancer_cases.csv                    29,253 bytes
  [OK] binary_2phase_missed_melanomas.csv                       24,857 bytes
  [OK] binary_2phase_missed_bcc_cases.csv                        4,619 bytes
  [OK] binary_2phase_evaluation_summary.csv                        802 bytes

All required files present.


## Section 14 - Final Summary (Copy-Paste Ready)

In [18]:
from IPython.display import display

print("=" * 74)
print("  11_pytorch_binary_2phase_final_evaluation -- FINAL SUMMARY")
print("=" * 74)

print(f"\n 1. torch version      : {torch.__version__}")
print(f" 2. CUDA available     : {CUDA_AVAILABLE}")
print(f" 3. GPU name           : {GPU_NAME}")
print(f" 4. Device used        : {DEVICE}")

print(f"\n 5. Model checkpoint   : {MODEL_PATH.name}")
print(f"    Checkpoint epoch   : {ckpt_epoch}")
print(f"    Checkpoint phase   : {ckpt_phase}")
print(f"    Checkpoint pr_auc  : {ckpt_pr_auc}")

print(f"\n 6. Validation rows    : {len(dfs['val']):,}")
print(f" 7. Test rows          : {len(dfs['test']):,}")

print(f"\n 8. Test binary counts:")
bc = dfs["test"]["binary_label"].value_counts()
print(f"      non_cancer  : {int(bc.get(0,0)):,}")
print(f"      cancer_risk : {int(bc.get(1,0)):,}")
print(f"      (MEL={int(dfs['test']['final_authoritative_label'].eq('MEL').sum()):,}  "
      f"BCC={int(dfs['test']['final_authoritative_label'].eq('BCC').sum()):,})")

print(f"\n 9. Test ROC-AUC       : {test_roc_auc:.5f}")
print(f"10. Test PR-AUC        : {test_pr_auc:.5f}")

print(f"\n11. Selected thresholds (from validation):")
for name, t in THRESHOLDS.items():
    print(f"      {name:<16} : {t:.4f}")

# Youden J metrics
yj = test_metrics_df[test_metrics_df["threshold_name"]=="youden_j"].iloc[0]
print(f"\n12. Test metrics at Youden J threshold ({THRESHOLDS['youden_j']:.4f}):")
print(f"      accuracy           : {yj['accuracy']:.5f}")
print(f"      cancer recall      : {yj['cancer_recall']:.5f}")
print(f"      specificity        : {yj['specificity']:.5f}")
print(f"      cancer precision   : {yj['cancer_precision']:.5f}")
print(f"      cancer F1          : {yj['cancer_f1']:.5f}")
print(f"      missed cancer (FN) : {int(yj['missed_cancer_count'])}")
print(f"      missed MEL         : {int(yj.get('missed_mel_count', -1))}")
print(f"      missed BCC         : {int(yj.get('missed_bcc_count', -1))}")

# High-recall metrics
hr = test_metrics_df[test_metrics_df["threshold_name"]=="high_recall"].iloc[0]
print(f"\n13. Test metrics at high-recall threshold ({THRESHOLDS['high_recall']:.4f}):")
print(f"      cancer recall      : {hr['cancer_recall']:.5f}")
print(f"      specificity        : {hr['specificity']:.5f}")
print(f"      cancer precision   : {hr['cancer_precision']:.5f}")
print(f"      cancer F1          : {hr['cancer_f1']:.5f}")
print(f"      missed cancer (FN) : {int(hr['missed_cancer_count'])}")
print(f"      missed MEL         : {int(hr.get('missed_mel_count', -1))}")
print(f"      missed BCC         : {int(hr.get('missed_bcc_count', -1))}")

print(f"\n14. Output file verification:")
for p in required_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"      [{status}] {p.name:<50} {size:>10,} bytes")

print(f"\n15. Confirmations:")
print(f"      No training performed                  : True")
print(f"      Validation used for threshold selection: True")
print(f"      Test used only for final evaluation    : True")
print(f"      Images modified                        : False")
print(f"      Manifests modified                     : False")

print(f"\n> Medical note: This model estimates cancer-risk probability for")
print(f"> screening support. It is not a clinical diagnosis.")

print(f"\nAll threshold results (test set):")
display(test_metrics_df[["threshold_name","threshold","cancer_recall",
    "specificity","cancer_precision","cancer_f1",
    "balanced_accuracy","roc_auc","pr_auc",
    "missed_cancer_count","missed_mel_count","missed_bcc_count",
    "FN","FP","TN","TP"]])

print("=" * 74)


  11_pytorch_binary_2phase_final_evaluation -- FINAL SUMMARY

 1. torch version      : 2.6.0+cu124
 2. CUDA available     : True
 3. GPU name           : NVIDIA GeForce RTX 4060 Laptop GPU
 4. Device used        : cuda

 5. Model checkpoint   : best_model_binary_2phase.pt
    Checkpoint epoch   : 17
    Checkpoint phase   : phase2_finetune
    Checkpoint pr_auc  : 0.90232

 6. Validation rows    : 3,012
 7. Test rows          : 3,045

 8. Test binary counts:
      non_cancer  : 1,894
      cancer_risk : 1,151
      (MEL=639  BCC=512)

 9. Test ROC-AUC       : 0.92758
10. Test PR-AUC        : 0.90013

11. Selected thresholds (from validation):
      default_050      : 0.5000
      max_f1           : 0.6300
      youden_j         : 0.4219
      high_recall      : 0.1700

12. Test metrics at Youden J threshold (0.4219):
      accuracy           : 0.84860
      cancer recall      : 0.84796
      specificity        : 0.84900
      cancer precision   : 0.77338
      cancer F1          : 0.80

,threshold_name,threshold,cancer_recall,specificity,cancer_precision,cancer_f1,balanced_accuracy,roc_auc,pr_auc,missed_cancer_count,missed_mel_count,missed_bcc_count,FN,FP,TN,TP
0,default_050,0.5000,0.82711,0.86959,0.79399,0.81021,0.84835,0.92758,0.90013,199,167.0,32.0,199,247,1647,952
1,max_f1,0.6300,0.80191,0.89810,0.82706,0.81429,0.85001,0.92758,0.90013,228,189.0,39.0,228,193,1701,923
2,youden_j,0.4219,0.84796,0.84900,0.77338,0.80895,0.84848,0.92758,0.90013,175,148.0,27.0,175,286,1608,976
3,high_recall,0.1700,0.92007,0.71700,0.66395,0.77130,0.81854,0.92758,0.90013,92,83.0,9.0,92,536,1358,1059


## Section 15 - Completion Summary

**11_pytorch_binary_2phase_final_evaluation is complete.**

**What was accomplished:**
- `best_model_binary_2phase.pt` loaded (no retraining).
- Inference run on validation and test sets with eval-only transform.
- Four thresholds selected from validation probabilities only:
  default (0.50), max F1, Youden J, and high-recall (≥ 0.90 cancer recall).
- Test set evaluated once at all four thresholds.
- Medical error files saved: missed cancers, missed MELanomas, missed BCC.
- Confusion matrix, ROC curve, and PR curve saved.

**What was deliberately deferred:**
- Multiclass evaluation (separate notebook).
- Calibration / reliability diagrams.
- Grad-CAM / explainability.
- Clinical deployment pipeline.

> **Medical note:** This model estimates cancer-risk probability for screening
> support. It is not a clinical diagnosis.
